# Med-Tracker — Med7 NER quickstart (Colab)

Extracts medication entities from prescription-label text using **Med7** (`en_core_med7_lg`).
Med7 is a spaCy NER model, **not** a generative LLM — that is deliberate (see the project spec).

> Real prescription data should be run locally, not in Colab. Use synthetic labels here.

In [ ]:
# 1. Install Med7 (pulls a compatible spaCy). If Colab warns about the spaCy
#    version: Runtime > Restart session, then run this cell again.
!pip -q install "en-core-med7-lg @ https://huggingface.co/kormilitzin/en_core_med7_lg/resolve/main/en_core_med7_lg-1.1.0-py3-none-any.whl" python-dateutil

In [ ]:
# 2. Load the model
import spacy
nlp = spacy.load("en_core_med7_lg")
print(spacy.__version__, nlp.pipe_names)

In [ ]:
# 3. Try it on a synthetic label
label_text = """GOODHEALTH PHARMACY   (555) 123-4567
Rx 4820193                Date filled: 08/01/2026

METFORMIN HCL 500 MG TABLET
Take 1 tablet by mouth twice daily with meals.

Qty: 60          Days supply: 30
"""

doc = nlp(label_text)
for ent in doc.ents:
    print(f"{ent.label_:<10} {ent.text!r}")

In [ ]:
# 4. Deterministic bits Med7 does NOT label: fill date, days supply -> refill date
import re, datetime as dt
from dateutil import parser as dateparser

def refill_date(text):
    d = re.search(r"date\s*filled\s*[:\-]?\s*([0-9/\-.]+)", text, re.I)
    s = re.search(r"day(?:s)?\s*supply\s*[:\-]?\s*(\d{1,3})", text, re.I)
    if not (d and s):
        return None
    start = dateparser.parse(d.group(1)).date()
    return (start + dt.timedelta(days=int(s.group(1)))).isoformat()

print("refill date:", refill_date(label_text))  # arithmetic, never predicted

## Next: OCR a real photo

```python
!apt-get -qq install tesseract-ocr && pip -q install pytesseract
from google.colab import files; up = files.upload()
import pytesseract; from PIL import Image, ImageOps
img = ImageOps.autocontrast(Image.open(next(iter(up))).convert('L'))
text = pytesseract.image_to_string(img)
print(text); doc = nlp(text)
for ent in doc.ents: print(ent.label_, ent.text)
```

For the full pipeline (OCR → NER → human confirm → profile) use `med7_pipeline.py` locally.